# Download & Tổ chức FAIDSet
Dataset: `ngocminhta/FAIDSet` — HuggingFace
```
raw_data/
├--- eng/  AI/  human/
└--- vi/   AI/  human/
```

In [1]:
# --- Cell 1: Cài thư viện ---
!pip install huggingface_hub langdetect -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# --- Cell 2: Download JSONL vào thư mục tạm ---
from pathlib import Path
from huggingface_hub import hf_hub_download

TMP  = Path("tmp_download")
TMP.mkdir(exist_ok=True)

SPLITS = {"train": "train.jsonl", "valid": "valid.jsonl", "test": "test.jsonl"}
raw_files = {}
for split, filename in SPLITS.items():
    path = hf_hub_download(
        repo_id="ngocminhta/FAIDSet",
        filename=filename,
        repo_type="dataset",
        local_dir=str(TMP)
    )
    raw_files[split] = Path(path)
    print(f"Downloaded {split}: {path}")

train.jsonl:   0%|          | 0.00/49.6M [00:00<?, ?B/s]

Downloaded train: tmp_download\train.jsonl


valid.jsonl:   0%|          | 0.00/10.1M [00:00<?, ?B/s]

Downloaded valid: tmp_download\valid.jsonl


test.jsonl:   0%|          | 0.00/10.1M [00:00<?, ?B/s]

Downloaded test: tmp_download\test.jsonl


In [3]:
# --- Cell 3: Phân loại, lưu file, xóa thư mục tạm ---
import json, shutil
from pathlib import Path
from collections import Counter
from langdetect import detect, LangDetectException

BASE = Path("raw_data")
for d in ["eng/AI", "eng/human", "vi/AI", "vi/human"]:
    (BASE / d).mkdir(parents=True, exist_ok=True)

# Đọc tất cả splits
all_records = []
for path in raw_files.values():
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                all_records.append(json.loads(line))
print(f"Tổng số record: {len(all_records):,}")

def get_kind(record):
    label = str(record.get("label", "")).lower()
    label = label.replace("\u2013", "-").replace("\u2014", "-")
    if label == "human-written": return "human"
    if label == "llm-generated":  return "AI"
    return None

def get_lang(text):
    try:
        lang = detect(text[:500])
        return {"vi": "vi", "en": "eng"}.get(lang)
    except LangDetectException:
        return None

# Phân loại và lưu
counts, skipped = Counter(), Counter()
for idx, record in enumerate(all_records):
    kind = get_kind(record)
    if not kind:    skipped["collaborative"] += 1; continue
    text = record.get("text", "").strip()
    if not text:    skipped["empty"] += 1;         continue
    lang = get_lang(text)
    if not lang:    skipped["other_lang"] += 1;    continue

    (BASE / lang / kind / f"{idx:06d}.txt").write_text(text, encoding="utf-8")
    counts[(lang, kind)] += 1
    if (idx + 1) % 5000 == 0:
        print(f"  {idx+1:,}/{len(all_records):,}...")

# Xóa thư mục tạm
shutil.rmtree(TMP)
print(f"  Đã xóa {TMP}/")

print("\nHoàn thành!")
for (lang, kind), c in sorted(counts.items()):
    print(f"  raw_data/{lang}/{kind}/  →  {c:,} files")
print(f"\nBỏ qua: {dict(skipped)}")

Tổng số record: 85,683
  20,000/85,683...
  25,000/85,683...
  30,000/85,683...
  35,000/85,683...
  40,000/85,683...
  60,000/85,683...
  70,000/85,683...
  Đã xóa tmp_download/

Hoàn thành!
  raw_data/eng/AI/  →  10,423 files
  raw_data/eng/human/  →  7,161 files
  raw_data/vi/AI/  →  9,161 files
  raw_data/vi/human/  →  13,066 files

Bỏ qua: {'collaborative': 45846, 'other_lang': 26}
